In [1]:
from nsga2.estimator import NSGAIIRegressor
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/lexicase_paper/d_airfoil.txt', sep=',')

# DEAP interface requires X and y to be numpy arrays, not pandas dataframes
X = df.drop('label', axis=1).values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y)

estimator = NSGAIIRegressor(**{
    'pop_size'        : 40, 
    'max_gen'         : 50,
    'max_depth'       : 7,  # 8
    'max_size'        : 2**7, # 75
    'objectives'      : ['error', 'size'],
    'initialization'  : 'uniform',
    'pick_criteria'   : 'error', # error, MCDM
    'validation_size' : 0.33,
    'simplify'        : True,
    
    # Either you use smart variation (just 1 cx and 1 mutation)
    'smart_variation' : True,

    # Or you use mabs (4 mutations)
    'use_mab'         : False,
    'use_context'     : False,

    'simplification_method' : 'bottom_up',
    'simplification_tolerance' : 1e-0,
    'verbosity'       : 1,
    'survival'       : 'tournament'
}).fit(X_train, y_train)

hashtable will have dimensions 256 x 755
starting to index. 755, 0
starting to index. 755, 1
starting to index. 755, 2
starting to index. 755, 3
starting to index. 755, 4
starting to index. 755, 5
initialized 6 keys
gen	evals	best_size	best_error	n_simplifications	n_new_hashes	avg train error	avg train size	avg val error	avg val size	med train error	med train size	med val error	med val size	std train error	std train size	std val error	std val size	min train error	min train size	min val error	min val size	max train error	max train size	max val error	max val size
0  	40   	11       	-98.3577  	84               	233         	               	              	             	            	               	              	             	            	               	              	             	            	               	              	             	            	               	              	             	            
1  	40   	4        	-46.5416  	62               	74          	               	  

In [2]:
# Should not give erros even without mabs
pd.DataFrame(estimator.variator.mab.pull_history).iloc[:10]

,t,arm,reward,update,delta_error,gen
0,0,cx,0.0,0,"[-3993.2359865031162, -0.0]",1
1,1,lsh_mutate,1.0,0,"[2632.360448548345, -0.0]",1
2,2,cx,1.0,0,"[403.71786479298044, -8.0]",1
3,3,lsh_mutate,1.0,0,"[14009.258059018146, -2.0]",1
4,4,subtree,1.0,0,"[1421.1276847097652, 1.0]",1
5,5,subtree,1.0,0,"[8186.466479526169, -2.0]",1
6,6,subtree,1.0,0,"[125289786421.98878, 3.0]",1
7,7,subtree,0.0,0,"[-10473.2275525535, 25.0]",1
8,8,lsh_mutate,0.0,0,"[-16.053671132724048, 4.0]",1
9,9,lsh_mutate,1.0,0,"[3249.8045829370094, -12.0]",1


In [3]:
pd.DataFrame(estimator.variator.mab.pull_history)['arm'].value_counts().sort_values()

lsh_mutate    644
cx            656
subtree       660
Name: arm, dtype: int64

In [4]:
pd.DataFrame(estimator.variator.mab.pull_history).groupby('arm')['reward'].value_counts().sort_values()

arm         reward
subtree     0.0        47
cx          0.0        69
lsh_mutate  0.0       118
subtree     0.5       126
cx          0.5       231
lsh_mutate  0.5       252
            1.0       274
cx          1.0       356
subtree     1.0       487
Name: reward, dtype: int64

In [5]:
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error as mse

model      = str(estimator.best_estimator_).replace("ARG", "x_")
size       = len(estimator.best_estimator_)
complexity = size
depth      = estimator.best_estimator_.height

print(model)
print(size)
print(complexity)
print(depth)

for metric, fn, (data_X, data_y) in [
    ('train_r2',  r2_score, (X_train, y_train)),
    ('test_r2',   r2_score, (X_test,  y_test )),
    ('train_mse', mse,      (X_train, y_train)),
    ('test_mse',  mse,      (X_test,  y_test )),
]:
    score = np.nan
    try:
        score = fn(estimator.predict(data_X), data_y)
        print(f"{metric} : {score}")
    except ValueError:
        print(f"(Failed to calculate {metric}")

print("arch size", len(estimator.archive_))
for ind in estimator.archive_:
    print(ind.fitness, ind)

square(subtract(11.453789094437035, mul4(add(absolute(log(x_2)), add(sqrt(x_1), sqrt(x_0))), add(x_4, x_2), arctan(exp(sin(log1p(square(cos(subtract(square(mul4(1.0000431429079308, sin(x_3), add3(x_2, 96.91252274232642, x_2), add3(-62.54083835830429, x_3, x_4))), x_0))))))), add(add(x_4, x_4), 0.01996099701225112))))
42
42
13
train_r2 : 0.23101835189257558
test_r2 : 0.15989850600284383
train_mse : 25.200341081840342
test_mse : 23.944261531451325
arch size 3
(23.163108105958827, 52.0) square(subtract(11.458361997494045, mul4(add(absolute(log(ARG2)), add(sqrt(ARG1), sqrt(ARG0))), add(ARG4, ARG2), arctan(exp(sin(log1p(square(cos(subtract(square(mul4(1.0000778508300425, minimum(expm1(minimum(arccos(ARG2), arctan(ARG2))), add(expm1(log(ARG2)), ARG4)), add3(ARG2, 96.9091395167415, ARG2), add3(-62.540838441876666, ARG3, ARG4))), ARG0))))))), add(add(ARG4, ARG4), 0.020484518384039512))))
(23.590566609501288, 42.0) square(subtract(11.453789094437035, mul4(add(absolute(log(ARG2)), add(sqrt(ARG1)

In [6]:
if False:
    print( len(list(estimator.simplifier.pop_hash.keys())) )

    n_keys =  len(list(estimator.simplifier.pop_hash.keys()))

    for key in list(estimator.simplifier.pop_hash.keys())[:n_keys]:
        print(key)
        for ind in estimator.simplifier.pop_hash[key]:
            print(" -", ind)

In [7]:
from sklearn.metrics import mean_squared_error
from numpy import (array, dot, arccos, clip)
from numpy.linalg import norm
import pandas as pd

frames = []
for key in list(estimator.variator.variator_.pop_hash.keys()):
    # print(key)
    for ind in estimator.variator.variator_.pop_hash[key]:
        # print(" -", ind)
        
        toolbox = estimator.toolbox_
        expr = toolbox.compile(expr=ind)
    
        pred = np.nan_to_num([expr(*x) for x in X_train])
        rmse_train = mean_squared_error(pred, y_train, squared=False)
        cos_train = arccos(clip(dot(pred,y_train)/norm(pred)/norm(y_train), -1, 1))

        pred = np.nan_to_num([expr(*x) for x in X_test])
        rmse_test = mean_squared_error(pred, y_test, squared=False)
        cos_test = arccos(clip(dot(pred,y_test)/norm(pred)/norm(y_test), -1, 1))
        
        frames.append({
            'key':key,
            'simplified_to':str(estimator.variator.variator_.pop_hash[key][0]).replace("ARG", "x_"),
            'expression':str(ind).replace("ARG", "x_"),
            'rmse_train':rmse_train,
            'cos_train':cos_train,
            'rmse_test':rmse_test,
            'cos_test':cos_test})

df = pd.DataFrame.from_records(frames)
df

<string>:1: RuntimeWarning: overflow encountered in exp
<string>:1: RuntimeWarning: overflow encountered in exp
<string>:1: RuntimeWarning: divide by zero encountered in log
<string>:1: RuntimeWarning: divide by zero encountered in log
<string>:1: RuntimeWarning: divide by zero encountered in log
<string>:1: RuntimeWarning: divide by zero encountered in log
<string>:1: RuntimeWarning: divide by zero encountered in log
<string>:1: RuntimeWarning: divide by zero encountered in log
<string>:1: RuntimeWarning: overflow encountered in exp
<string>:1: RuntimeWarning: overflow encountered in exp
<string>:1: RuntimeWarning: overflow encountered in exp
<string>:1: RuntimeWarning: overflow encountered in exp
<string>:1: RuntimeWarning: overflow encountered in expm1
<string>:1: RuntimeWarning: overflow encountered in expm1
<string>:1: RuntimeWarning: overflow encountered in expm1
<string>:1: RuntimeWarning: overflow encountered in expm1
<string>:1: RuntimeWarning: overflow encountered in expm1
<s

,key,simplified_to,expression,rmse_train,cos_train,rmse_test,cos_test
0,0,1.0,1.0,124.120082,0.055612,123.750874,0.053829
1,1,x_0,x_0,4172.096512,0.846675,4252.227799,0.867413
2,1,x_0,"add4(x_0, 91.047762040034, x_2, x_0)",8486.867011,0.838876,8643.882588,0.859641
3,1,x_0,"maximum(x_1, x_0)",4172.096512,0.846675,4252.227799,0.867413
4,1,x_0,"add(x_0, x_0)",8425.496790,0.846675,8583.542052,0.867413
...,...,...,...,...,...,...,...
8957,1840,exp(sin(log1p(square(cos(subtract(square(mul4(...,exp(sin(log1p(square(cos(subtract(square(mul4(...,123.650251,0.224497,123.292968,0.222854
8958,1841,sin(log1p(square(cos(subtract(square(mul4(1.00...,sin(log1p(square(cos(subtract(square(mul4(1.00...,124.758744,0.567672,124.397490,0.572768
8959,1842,log1p(square(cos(subtract(square(mul4(1.000043...,log1p(square(cos(subtract(square(mul4(1.000043...,124.738853,0.579562,124.378482,0.585003
8960,1843,square(cos(subtract(square(mul4(1.000043142907...,square(cos(subtract(square(mul4(1.000043142907...,124.612404,0.620532,124.256967,0.627730


In [8]:
display(df.describe())

A = np.maximum(df['x0'], df['x4']+29.657).values
B = df['label'].values

print(np.std(A) * (A - np.mean(A))[:5])
print(np.std(B) * (B - np.mean(B))[:5])

,rmse_train,cos_train,rmse_test,cos_test
count,8962.000000,5722.000000,8962.000000,5729.000000
mean,inf,0.923311,inf,0.922412
std,NaN,0.703505,NaN,0.706418
min,4.858592,0.038788,4.732623,0.037596
25%,124.463585,0.327401,124.081794,0.322992
50%,125.118523,0.848931,124.749414,0.867412
75%,125.450676,1.287019,125.072048,1.296325
max,inf,3.089048,inf,3.089388


KeyError: 'x0'